# 08 | Persistence and Custom Extensions

The final notebook covers model persistence and the three main extension points: custom losses, optimizers, and contraction strategies. It ends with a practical source-reading map and version-specific cautions.


In [1]:
from pathlib import Path
import sys

# This works whether Jupyter starts in the repository root or in notebooks/.
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "tneq_qc").is_dir() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/yuch3n/Documents/Code/Github/tneq-qc


In [2]:
from pathlib import Path
import tempfile

from tneq_qc import BackendFactory, QCTN

backend = BackendFactory.create_backend("pytorch", device="cpu", dtype="float32")
graph = """\
-2-A-2-
-2-A-2-"""
model = QCTN(graph, backend=backend).auto_init(orthogonal=True)


## 1. Save and load safetensors

`save_cores()` stores effective values with scale applied, together with readable core names, batch flags, and optional metadata. Loading reconstructs suitable `TNTensor` objects.

This example uses a temporary directory so it does not leave tutorial artifacts in the repository.


In [3]:
temp_dir = Path(tempfile.mkdtemp(prefix="tneq_qc_tutorial_"))
save_path = temp_dir / "demo.safetensors"

model.save_cores(save_path, metadata={
    "tutorial": "08",
    "description": "beginner save/load example",
})
print("Saved to:", save_path)


Saved to: /var/folders/q0/f_mhdfv55nv4vm0ym3hlgc8m0000gn/T/tneq_qc_tutorial_xr8oq2y4/demo.safetensors


In [4]:
reloaded = QCTN(graph, backend=backend)
metadata = reloaded.load_cores(save_path)

print("Metadata:", metadata)
for name in model.cores:
    before = model[name].numpy()
    after = reloaded[name].numpy()
    print(name, "maximum absolute error =", abs(before - after).max())

pretrained = QCTN.from_pretrained(graph, save_path, backend=backend)
print("from_pretrained result:", pretrained)


Metadata: {'tutorial': '08', 'description': 'beginner save/load example', '_core_names': '{"A": "A"}', '_core_has_batch': '{"A": false}'}
A maximum absolute error = 2.9802322e-08
from_pretrained result: QCTN(nqubits=2, cores=[A(2, 2, 2, 2)])


## 2. Register a custom loss

A custom loss inherits `BaseLoss` and implements `compute(result, target, backend)`. Always use the effective values of `TNTensor` inputs.


In [5]:
from tneq_qc.losses import BaseLoss, LossRegistry, register_loss


@register_loss("tutorial_l2")
class TutorialL2Loss(BaseLoss):
    def compute(self, result, target, backend):
        result_value = result.tensor * result.scale
        target_value = target.tensor * target.scale
        difference = result_value - target_value
        return backend.mean(backend.abs_square(difference))


print(LossRegistry.available())


['diagonal_mse', 'fidelity', 'mae', 'mse', 'nll', 'tutorial_l2']


## 3. Register a custom optimizer

Inherit `OptimizerBase` and implement `update_raw_params()`. Registration makes the class available through `create_optimizer()`.


In [6]:
from tneq_qc.optim import OptimizerBase, register_optimizer


class TutorialGradientDescent(OptimizerBase):
    method = "tutorial_gd"

    def update_raw_params(self, params, grads, state, hyperparams):
        lr = hyperparams.get("learning_rate", 0.01)
        updated = [param - lr * grad for param, grad in zip(params, grads)]
        return updated, state


register_optimizer("tutorial_gd", TutorialGradientDescent)


## 4. Custom contraction strategy interface

A strategy implements:

- `name`;
- `check_compatibility()`;
- `estimate_cost()`;
- `get_compute_function()`.

This is an advanced extension point. Study `EinsumStrategy` and reuse QCTN's generated index information before attempting custom index management.


In [7]:
from tneq_qc import ContractionStrategy, register_contraction_strategy


class TutorialStrategySkeleton(ContractionStrategy):
    @property
    def name(self):
        return "tutorial_skeleton"

    def check_compatibility(self, qctn, shapes_info):
        return False  # Do not select this unfinished teaching skeleton.

    def estimate_cost(self, qctn, shapes_info):
        return float("inf")

    def get_compute_function(self, qctn, shapes_info, backend):
        raise NotImplementedError("Implement a real contraction function first.")


register_contraction_strategy(TutorialStrategySkeleton())


## 5. Suggested source-reading order

1. `tneq_qc/core/qctn.py` — object model, parameters, concat, trace, and Hermitian views;
2. `tneq_qc/core/tn_tensor.py` — scale and reference semantics;
3. `tneq_qc/core/_qctn_graph.py` — ASCII graph parsing;
4. `tneq_qc/contractor/row_priority_strategy.py` — main contraction path;
5. `tneq_qc/core/engine_common.py` — forward, loss, and gradients;
6. `tneq_qc/modules/app.py` — BornMachine composition;
7. `tneq_qc/distributed/` — partitioning and communication, after mastering the single-process path.

## 6. Current-version cautions

- Use `BornMachine`, not the older README name `Quadratic`.
- Use `State`, not `CircuitState`.
- The engine argument is `strategy=...`, not `strategy_mode=...`.
- `StepLRScheduler` accepts a list of `(step, learning_rate)` pairs.
- The registered built-in strategies are `row_priority` and `einsum_default`.
- The old `Trainer` is deprecated; write direct training loops.
- Distributed functionality is experimental. Validate the single-process result before testing multiprocess gradients.

## 7. Good next engineering tasks

- Add `pyproject.toml` and a locked dependency specification.
- Run the complete test suite and execute these notebooks in CI.
- Reconcile README and docs examples with the current API.
- Add numerical equivalence tests for einsum and row-priority contraction.
- Verify distributed forward and backward results against the single-process engine.

You have now followed the complete path from topology, tensor representation, and composition through training, sampling, persistence, and extension.
